# Lab 10: Trees, then networks begin

Two questions for the night. What if the model cuts the data into boxes instead of drawing a line? And what if the model learns its own features?

We start with one split on ten diamonds, grow and prune a tree on house prices, and grade a classification tree with the Gini index. Then we average many trees (bagging and random forests) and add trees one at a time (boosting). In the last section a small neural network, with weights set by hand, separates the circle that a straight line could not.

1. One split by hand: diamonds
2. Growing and pruning a tree: Saratoga houses
3. Classification trees and the Gini index
4. Bagging and random forests
5. Boosting, briefly
6. Networks begin: the circle, a hidden layer, a forward pass (with the computing corner: tensor shapes, matrix products, CPU against GPU)
7. Exit problem and at home


## Route

**Three outcomes.** After this lab you can (1) compute the SSE of a split and the Gini index of a node, and say what bagging, random forests and boosting each change; (2) run the forward pass of a small ReLU network for one point and for a whole batch as one matrix product, with the shape of every array labelled; (3) say what a matrix product costs and why the best device depends on the size of the job.

**Core sections for class (about 160 minutes).** 1. One split by hand (20). 2. Growing and pruning a tree (25). 3. Classification trees and Gini (15). Break (10). 4. Bagging and random forests (20). 5. Boosting (15). 6. Networks begin: the circle and the forward pass, with Exercise 10.5 (25). Computing corner: why matrix shapes and big operations decide speed on a CPU and a GPU (20). Exit problem (10).

**Work to do.** All sections above are core. Every graded exercise is required: 10.1 to 10.5 (best done in class, each sits right after the example it practises) and 10.6 (at home). Plan for about 60 to 90 minutes after class (a planning target, not a measurement). If `torch` is not installed, the CPU against GPU cell will not run; nothing graded depends on it.

**Due before class on 18 November.** Submit the notebook on Canvas.

Run All stops at the first empty exercise check; that is expected. Fill the exercises, then Restart and Run All before you submit.


## How to work

- Run the cells from the top. Before each cell marked **Predict**, write down your guess, then run it.
- **Your turn** cells run as given. Change the line marked `# change this` and read what happens.
- **Check yourself** answers are hidden. Open them after you have tried.
- When a **Predict** prompt asks for a guess, write it down and talk to a neighbour for a minute before you run the cell.
- **Lab exercises** are graded. Each has a task cell, a code cell for you, and a check cell. A check prints `ok` when your answer is right.
- Allowed libraries: `numpy`, `pandas`, `matplotlib`, `scikit-learn`. The CPU against GPU cell also uses `torch`.
- The first cell downloads four data files into `data/`: `diamonds.csv` (a random sample of 5,000 rows of the classic diamonds data), `SaratogaHouses.csv`, `credit.csv`, and `circle.csv`.
- The book's Boston housing data is not available, so we use the Saratoga houses (New York State, 1,728 homes) for the regression tree.


**Matching online lab.** [Neural Net Playground](https://vsokolov.org/labs/neural-net-lab.html) goes with Section 6.


In [ ]:
from hashlib import sha256
from pathlib import Path
from time import perf_counter
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text

DATA = Path("data")
DATA.mkdir(exist_ok=True)
BASE = "https://vsokolov.org/courses/files/600/data"
for name in ("diamonds.csv", "SaratogaHouses.csv", "credit.csv", "circle.csv"):
    path = DATA / name
    if not path.exists():
        urlretrieve(f"{BASE}/{name}", path)

plt.rcParams["figure.figsize"] = (5, 3.5)


def check(name, value, expected, ndigits=6):
    if value is None:
        raise AssertionError(f"{name} is still None: define it in the exercise cell")
    try:
        x = float(value)
    except (TypeError, ValueError):
        raise AssertionError(f"{name} must be a plain number, not {type(value).__name__}")
    if not np.isfinite(x):
        raise AssertionError(f"{name} is not a finite number")
    s = f"{round(x, ndigits):.{ndigits}f}"
    got = sha256(f"aii600-lab10|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"{name} does not match the right value (yours: {x:.6g}). Re-read the task and its inputs."
    print("ok ", name)


def check_array(name, value, expected, shape, ndigits=6):
    # compares the whole array (rounded to 6 digits), not just a sum
    if value is None:
        raise AssertionError(f"{name} is still None: define it in the exercise cell")
    arr = np.asarray(value, dtype=float)
    if arr.shape != tuple(shape):
        raise AssertionError(f"{name} has shape {arr.shape}, but the task asks for shape {tuple(shape)}")
    if not np.all(np.isfinite(arr)):
        raise AssertionError(f"{name} contains values that are not finite")
    key = (np.round(arr, ndigits) + 0.0).tobytes()
    got = sha256(f"aii600-lab10|{name}|".encode() + key).hexdigest()[:16]
    assert got == expected, f"{name} has the right shape but some values are wrong (check the axis and the bias)"
    print("ok ", name)


def best_of(f, repeat=3):
    # run f a few times and return the fastest time in seconds
    times = []
    for _ in range(repeat):
        t0 = perf_counter()
        f()
        times.append(perf_counter() - t0)
    return min(times)


def rmse(a, b):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))


## 1. One split by hand: diamonds

A **regression tree** predicts a number by asking yes-or-no questions about the inputs. Each question is a **split**: "is carat below 0.5?". Every diamond that lands in the same group (a **leaf**) gets the same prediction, the average price of the training diamonds in that group.

How does the tree pick a split? It tries candidate cut points and keeps the one with the smallest **SSE**, the sum of squared errors. For a group of prices, the SSE is the sum of (price minus the group mean) squared. The score of a split is the SSE of the left group plus the SSE of the right group.

Here are 10 diamonds drawn at random from the data (carat is the weight, price is in US dollars), sorted by carat.


In [ ]:
diamonds = pd.read_csv(DATA / "diamonds.csv")
ten = diamonds.sample(10, random_state=4)[["carat", "price"]].sort_values("carat").reset_index(drop=True)
print(ten.T.to_string())


**Predict:** we will score three cuts, carat below 0.5, carat below 0.68, and carat below 1.0. Which one gives the smallest SSE? Write your guess, then run the cell.


In [ ]:
def sse(values):
    values = np.asarray(values, float)
    return float(((values - values.mean()) ** 2).sum())


def split_sse(table, cut):
    left = table.price[table.carat < cut]
    right = table.price[table.carat >= cut]
    return sse(left) + sse(right)


sse_no_split = sse(ten.price)
print(f"no split: one mean for all 10 diamonds, SSE = {sse_no_split:,.0f}")
for cut in (0.5, 0.68, 1.0):
    left = ten.price[ten.carat < cut]
    right = ten.price[ten.carat >= cut]
    print(f"cut {cut:4.2f}: left n={len(left)} mean={left.mean():7.0f} SSE={sse(left):12,.0f} | "
          f"right n={len(right)} mean={right.mean():7.0f} SSE={sse(right):12,.0f} | total {split_sse(ten, cut):12,.0f}")
three_cuts = [0.5, 0.68, 1.0]
three_sse = [split_sse(ten, c) for c in three_cuts]
winner_cut = three_cuts[int(np.argmin(three_sse))]


**Reading.** With no split the SSE is 139,370,516. The cut at 1.0 wins among the three, with SSE 39,595,030. A good split puts the cheap diamonds on one side and the expensive ones on the other, so that each side is close to its own mean.


Now let the computer try every cut. The rule is: diamonds with carat < cut go left, the others go right. A cut never falls between two equal carat values, so we take the distinct carat values, sort them, and use the midpoints between neighbours as candidates. Then we ask scikit-learn for a tree with one split (a **stump**) and compare.


In [ ]:
values = np.sort(ten.carat.unique())
midpoints = (values[:-1] + values[1:]) / 2
all_sse = np.array([split_sse(ten, c) for c in midpoints])
best_cut_ten = float(midpoints[all_sse.argmin()])
winner_sse = None
stump = DecisionTreeRegressor(max_depth=1).fit(ten[["carat"]], ten.price)
sklearn_cut = float(stump.tree_.threshold[0])
print(f"{len(midpoints)} candidate cuts; best by hand: {best_cut_ten:.3f} with SSE {all_sse.min():,.0f}")
print(f"scikit-learn stump chose:   {sklearn_cut:.3f}")
print("leaf predictions (left, right):", stump.tree_.value[1:, 0, 0].round(0))


**Reading.** There are 8 candidate cuts. The best one is at carat 1.190, with SSE 25,035,238. Scikit-learn picks the same cut (1.190). The leaf predictions are the two group means.


**Check yourself**

1. Why is the candidate cut a midpoint and not a diamond's own carat value?
2. If every diamond in a leaf has the same price, what is the SSE of that leaf?

<details><summary>Answer</summary>

1. Any cut between two neighbouring carat values sends the same diamonds left and right, so the SSE is the same. The midpoint is a tidy choice. A cut equal to a diamond's carat is also fine, but then you must decide which side gets the tie.
2. Zero. Every price equals the leaf mean.

</details>


In [ ]:
small = diamonds.sample(8, random_state=5)[["carat", "price"]].reset_index(drop=True)
print(small.T.to_string())


### Exercise 10.1 (In class)

**Task.** Find the best split of a small table of 8 diamonds: try every midpoint between neighbouring distinct carat values and keep the one with the smallest total SSE.

**Inputs.** `small` (a DataFrame with columns `carat` and `price`), and the function `split_sse(table, cut)` from above.

**Define.** `best_cut` (a number, the winning cut) and `best_sse` (a number, its total SSE).

**Form of the answer.** Two numbers. Use the rule carat < cut goes left. Careful: some carat values appear twice, so use the distinct values (`np.unique`) and put a cut only between neighbouring distinct values.


In [ ]:
# Exercise 10.1: define best_cut, best_sse
best_cut = None
best_sse = None


In [ ]:
check("best_cut", best_cut, "8c73f39e8e2dec61")
check("best_sse", best_sse, "9a2563b9fc7ae1c3")


## 2. Growing and pruning a tree: Saratoga houses

A tree grows by repeating the split step. First it splits all the data. Then it splits each new group, and so on, until a stopping rule says stop. A tree that is allowed to grow without limit gives each house its own leaf. It fits the training data perfectly and generalizes badly.

The data: 1,728 homes in Saratoga County, New York (`price` in US dollars, plus size, age, land value, and other features). We turn text columns into 0 or 1 columns (dummy variables), keep 70% of the rows for training and 30% for validation, and compare errors. The error measure is the RMSE, the square root of the mean squared error, in dollars.


In [ ]:
homes = pd.read_csv(DATA / "SaratogaHouses.csv", index_col=0)
Xall = pd.get_dummies(homes.drop(columns="price"), drop_first=True).astype(float)
yall = homes.price.to_numpy()
order = np.random.default_rng(10).permutation(len(homes))
n_train = int(0.7 * len(homes))
Xtr, ytr = Xall.iloc[order[:n_train]], yall[order[:n_train]]
Xva, yva = Xall.iloc[order[n_train:]], yall[order[n_train:]]
baseline_rmse = rmse(yva, np.full(len(yva), ytr.mean()))
print("train rows", len(Xtr), "validation rows", len(Xva), "features", Xall.shape[1])
print(f"always predict the training mean: validation RMSE = {baseline_rmse:,.0f}")


The main knob is `max_depth`, the number of questions on the longest path. Depth 1 is a stump. We sweep it and watch both errors.


In [ ]:
depths = list(range(1, 16))
train_err, val_err = [], []
for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(Xtr, ytr)
    train_err.append(rmse(ytr, tree.predict(Xtr)))
    val_err.append(rmse(yva, tree.predict(Xva)))

best_depth = depths[int(np.argmin(val_err))]
plt.plot(depths, train_err, "C0-", label="training")
plt.plot(depths, val_err, "C3--", label="validation")
plt.axvline(best_depth, color="gray", lw=0.8)
plt.xlabel("max_depth (number of questions)")
plt.ylabel("RMSE (US dollars)")
plt.legend()
plt.tight_layout()
plt.show()
print(f"best depth {best_depth}: validation RMSE {min(val_err):,.0f}; training RMSE at depth 15: {train_err[-1]:,.0f}")


**Reading.** The training error keeps falling as the tree grows, down to 6,942 dollars at depth 15. The validation error falls until depth 4 (60,659 dollars) and then stops improving or gets worse. The baseline that ignores all features has 95,645. The gap between the two curves is overfitting. One caution: we chose the depth using these validation rows, so 60,659 is a little optimistic. A third set of houses, a test set used only once, would give an honest number. The same holds for the pruning alpha in Exercise 10.2.


In [ ]:
# Your turn: fit one tree with the depth below and read its validation RMSE.
# Target: find a depth whose validation RMSE is below the baseline printed above, and another that is worse than the best.
max_depth = 4  # change this
tree = DecisionTreeRegressor(max_depth=max_depth, random_state=0).fit(Xtr, ytr)
print(f"max_depth {max_depth}: {tree.get_n_leaves()} leaves, validation RMSE {rmse(yva, tree.predict(Xva)):,.0f}")
print(export_text(tree, feature_names=list(Xall.columns), max_depth=2))


**How to read `export_text`.** Each line is a question, such as `livingArea <= 1899.00`. The lines indented under a `<=` line are the yes branch. The lines under the matching `>` line are the no branch. `truncated branch of depth 2` means there are deeper splits that are hidden to keep the print short.

### Pruning

A second way to stop overfitting is to grow a big tree and then cut branches back. **Cost-complexity pruning** scores a tree by

SSE / n + alpha × (number of leaves), where n is the number of training rows.

This is scikit-learn's convention: it divides the SSE by n, so alpha is measured in squared dollars per leaf. (If you used the plain SSE, the price per leaf would be alpha × n. Do not mix the two.) The number alpha is a price per leaf. With alpha = 0 we keep the full tree. As alpha grows, the weakest branches are removed first, and the tree gets smaller. Scikit-learn lists the alpha values at which a branch disappears. We keep every 10th one to save time. The validation split picks the winner.


In [ ]:
path = DecisionTreeRegressor(random_state=0).cost_complexity_pruning_path(Xtr, ytr)
alphas = path.ccp_alphas[::10]
full_leaves = DecisionTreeRegressor(random_state=0).fit(Xtr, ytr).get_n_leaves()
print(f"{len(path.ccp_alphas)} alpha values; we try {len(alphas)} of them")
print(f"unpruned tree: {full_leaves} leaves; alpha runs from {alphas[0]:.0f} to {alphas[-1]:,.0f}")


**Reading.** The full tree has 1156 leaves. The training set has 1209 houses, so the tree has nearly one leaf per house. Pruning picks a much smaller tree. Exercise 10.2 does the choice.


### Exercise 10.2 (In class)

**Task.** Choose the pruning strength alpha on the validation split. For each alpha in `alphas`, fit `DecisionTreeRegressor(random_state=0, ccp_alpha=alpha)` on `Xtr, ytr`, compute the validation RMSE on `Xva, yva` (use `rmse`), and keep the alpha with the smallest validation RMSE. If two alphas tie, keep the first.

**Inputs.** `Xtr, ytr, Xva, yva, alphas, rmse`.

**Define.** `best_val_rmse` (a number, the smallest validation RMSE) and `best_leaves` (an integer, the number of leaves of the winning tree).

**Form of the answer.** Two numbers. The check allows a small tolerance (library numbers can differ a little between scikit-learn versions).


In [ ]:
# Exercise 10.2: define best_val_rmse, best_leaves
best_val_rmse = None
best_leaves = None


In [ ]:
assert best_val_rmse is not None and best_leaves is not None, "define best_val_rmse and best_leaves"
assert abs(best_val_rmse - 60191.31) / 60191.31 < 0.005, "best_val_rmse is not within 0.5% of the right value"
assert abs(best_leaves - 11) <= 3, "best_leaves is not within 3 of the right value"
print("ok  pruning")


### Computing corner: split search

How long does one split take? For one feature, a tree sorts the rows by that feature, then slides the cut from left to right, updating sums as it goes. The sort costs about n log n steps for n rows. The slide costs n. A tree must do this for every feature at every node, so a tree with p features costs about p × n log n per level. That is why big trees on big tables are slow.

The cell times a sort for growing n. Each time n grows 10 times, the time grows a little more than 10 times.


In [ ]:
rng_sort = np.random.default_rng(0)
sort_times = {}
for n_rows in (10**5, 10**6, 10**7):
    feature = rng_sort.standard_normal(n_rows)
    sort_times[n_rows] = best_of(lambda: np.sort(feature))
    print(f"sort {n_rows:>10,} numbers: {1000 * sort_times[n_rows]:8.1f} ms")
print(f"Saratoga has {Xall.shape[1]} features: one level of splits sorts about {Xall.shape[1]} columns of {len(Xtr)} rows.")


**Check yourself**

1. Which has the smaller training error: a depth-3 tree or a depth-10 tree? Which has the smaller validation error?
2. What does a larger alpha do to the number of leaves?

<details><summary>Answer</summary>

1. The depth-10 tree always has the smaller or equal training error. The validation error can be either; usually the depth-10 tree is worse on validation because it has fitted noise.
2. It reduces the number of leaves, because each leaf costs more.

</details>


## 3. Classification trees and the Gini index

For a yes-or-no outcome, a leaf predicts the most common class in the leaf. To pick splits we need an impurity measure instead of the SSE. The **Gini index** of a group is

Gini = 1 − (share of class 0)² − (share of class 1)².

A pure group (all one class) has Gini 0. A 50-50 group has Gini 0.5, the largest possible for two classes. A split is scored by the Gini of the two children, each weighted by its share of the rows. The tree keeps the split with the lowest weighted Gini.

The data: German credit, 1,000 loans, with `Default` equal to 1 if the loan went bad.


In [ ]:
credit = pd.read_csv(DATA / "credit.csv")
Xc = pd.get_dummies(credit.drop(columns="Default"), drop_first=True).astype(float)
yc = credit.Default.to_numpy()
order_c = np.random.default_rng(11).permutation(len(credit))
n_train_c = int(0.7 * len(credit))
Xc_tr, yc_tr = Xc.iloc[order_c[:n_train_c]], yc[order_c[:n_train_c]]
Xc_va, yc_va = Xc.iloc[order_c[n_train_c:]], yc[order_c[n_train_c:]]

# Gini by hand for one node: loans whose checking account status is A14 (no checking account)
node = credit[credit.checkingstatus1 == "A14"]
share_default = node.Default.mean()
gini_node_a14 = 1 - share_default**2 - (1 - share_default) ** 2
overall_share = credit.Default.mean()
gini_all = 1 - overall_share**2 - (1 - overall_share) ** 2
print(f"all loans: {len(credit)} rows, default share {overall_share:.3f}, Gini {gini_all:.3f}")
print(f"node A14:  {len(node)} rows, default share {share_default:.3f}, Gini {gini_node_a14:.3f}")


**Reading.** Across all loans the default share is 0.300, so the Gini is 0.420, close to the 0.5 maximum. The group with no checking account is purer: 394 loans, default share 0.117, Gini 0.206. Splitting off this group is a good first question.


### Exercise 10.3 (In class)

**Task.** Score the split "duration of the loan is more than 24 months" by weighted Gini. Compute the Gini of all loans (the parent), then the Gini of the two children (duration above 24 and duration at most 24), each weighted by its share of the loans.

**Inputs.** `credit` (columns `Default` and `duration`).

**Define.** `gini_parent` (a number) and `gini_after` (a number, the weighted sum of the two child Ginis).

**Form of the answer.** Two numbers. Use the formula above; do not call a library function.


In [ ]:
# Exercise 10.3: define gini_parent, gini_after
gini_parent = None
gini_after = None


In [ ]:
check("gini_parent", gini_parent, "5bd92bfd0eed286d")
check("gini_after", gini_after, "ba79366873a23574")


Now grow a small tree and compare it with logistic regression. The yardstick is the **AUC**, the area under the ROC curve: the chance that a random defaulter gets a higher score than a random non-defaulter, plus half the chance that the two scores are tied (0.5 is a coin flip, 1.0 is perfect). Ties are common for trees, because all loans in a leaf share one score. Both models are scored on the validation loans.


In [ ]:
clf = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xc_tr, yc_tr)
auc_tree = roc_auc_score(yc_va, clf.predict_proba(Xc_va)[:, 1])
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xc_tr, yc_tr)
auc_logit = roc_auc_score(yc_va, logit.predict_proba(Xc_va)[:, 1])
print(export_text(clf, feature_names=list(Xc.columns), max_depth=2))
print(f"validation AUC: tree (depth 3) {auc_tree:.3f}, logistic regression {auc_logit:.3f}")
better_model = "logistic regression" if auc_logit > auc_tree else "the tree"


**Reading.** The depth-3 tree has validation AUC 0.750 and logistic regression has 0.823, so logistic regression ranks the loans better here. The tree gives rules a clerk can read. How to read the printed tree: each line is a question. Lines under a `<=` line are the yes branch. For example, `checkingstatus1_A14 <= 0.50` asks whether the loan does not have status A14 (the dummy column equals 0). A split may give the same class on both sides (two `class: 0` leaves); it is kept because it still lowers the impurity. The logistic model usually ranks better when effects are smooth and add up. A tree can win when effects interact or jump.


**Check yourself**

1. A node has 30 non-defaulters and 10 defaulters. What is its Gini?
2. Why can a tree with only a handful of leaves give only a handful of different probabilities, and what does this do to the ROC curve?

<details><summary>Answer</summary>

1. The default share is 10/40 = 0.25, so Gini = 1 − 0.25² − 0.75² = 0.375.
2. Every loan in a leaf gets the same probability. With 8 leaves there are at most 8 distinct scores, so the ROC curve has only a few corners and is a coarse staircase.

</details>


## 4. Bagging and random forests

One deep tree is unstable: change a few rows and the tree changes a lot. **Bagging** (bootstrap aggregating) fixes this by averaging.

1. Draw a **bootstrap sample**: pick n rows from the n training rows with replacement. Some rows repeat, some are left out.
2. Grow a deep tree on that sample.
3. Repeat many times and average the predictions.

A **random forest** adds one more idea: at each split the tree may only look at a random subset of the features (here one third of them, `max_features=1/3`). With all features allowed at every split (`max_features=1.0`, scikit-learn's default for regression) the method is plain bagging. That makes the trees less alike, so averaging helps more.

First a picture with one feature, living area. We fit five trees (depth 10) on five bootstrap samples and average them. **Predict:** will the largest jump of the average of five trees be smaller or larger than that of one tree?


In [ ]:
area_tr = Xtr[["livingArea"]].to_numpy()
grid = np.linspace(area_tr.min(), area_tr.max(), 300).reshape(-1, 1)
rng_boot = np.random.default_rng(4)
one_tree = DecisionTreeRegressor(max_depth=10, random_state=0).fit(area_tr, ytr)
preds = []
for b in range(5):
    rows = rng_boot.integers(0, len(area_tr), len(area_tr))
    t = DecisionTreeRegressor(max_depth=10, random_state=b).fit(area_tr[rows], ytr[rows])
    preds.append(t.predict(grid))
bag5 = np.mean(preds, axis=0)

fig, ax = plt.subplots()
ax.scatter(area_tr, ytr / 1000, s=3, color="lightgray")
for p in preds:
    ax.plot(grid, p / 1000, lw=0.7, alpha=0.5, color="C0")
ax.plot(grid, one_tree.predict(grid) / 1000, "C3--", lw=1.5, label="one tree")
ax.plot(grid, bag5 / 1000, "C0-", lw=2, label="average of 5 bootstrap trees")
ax.set_xlabel("living area (square feet)")
ax.set_ylabel("price (thousand US dollars)")
ax.set_ylim(0, 800)
ax.legend()
plt.tight_layout()
plt.show()

jumps_one = float(np.abs(np.diff(one_tree.predict(grid))).max())
jumps_bag = float(np.abs(np.diff(bag5)).max())
print(f"largest single jump along the curve (US dollars): one tree {jumps_one:,.0f}, average of five {jumps_bag:,.0f}")


**Reading.** The average of five trees (solid) has smaller steps than one tree (dashed), because the five trees put their jumps at different places. It is still piecewise constant: an average of step functions is a step function, with smaller jumps, not a smooth curve. The largest single jump along the curve is 531,000 dollars for one tree and 262,400 for the average. With more bootstrap trees the jumps usually get smaller, but this is a tendency, not a guarantee for every sample.


### Out-of-bag error

A bootstrap sample leaves out about 37% of the rows (each row is missed with chance (1 − 1/n)ⁿ, which is about 1/e = 0.368 for large n). Those left-out rows are the **out-of-bag** (OOB) rows of that tree. To score a row, average only the trees that did not see it. This gives a validation-like error with no separate split.

Now a forest on all features, with 300 trees.


In [ ]:
rng_oob = np.random.default_rng(0)
left_out = np.mean([len(np.setdiff1d(np.arange(1000), rng_oob.integers(0, 1000, 1000))) / 1000 for _ in range(200)])
print(f"share of rows left out of one bootstrap sample (simulated, n=1000): {left_out:.3f}")

forest = RandomForestRegressor(n_estimators=300, max_features=1 / 3, oob_score=True, random_state=0, n_jobs=-1).fit(Xtr, ytr)
forest_val = rmse(yva, forest.predict(Xva))
best_tree_val = min(val_err)
print(f"one tree (best depth {best_depth}): validation RMSE {best_tree_val:,.0f}")
print(f"forest of 300 trees:          validation RMSE {forest_val:,.0f}, OOB R^2 {forest.oob_score_:.3f}")
val_r2 = 1 - np.mean((yva - forest.predict(Xva)) ** 2) / np.var(yva)
print(f"                              validation R^2 {val_r2:.3f}")
bag_all = RandomForestRegressor(n_estimators=300, max_features=1.0, random_state=0, n_jobs=-1).fit(Xtr, ytr)
bag_val = rmse(yva, bag_all.predict(Xva))
print(f"bagging (all features at every split): validation RMSE {bag_val:,.0f}")


**Reading.** A bootstrap sample leaves out about 0.367 of the rows. The forest has validation RMSE 51,438 against 60,659 for the best single tree. Bagging with all features at every split has validation RMSE 52,539. R² is the share of the variance of price that the model explains: 1 is perfect and 0 is no better than predicting the mean price. R² can even be negative for a very poor model. For this fitted forest the OOB R² is 0.659 and the validation R² is 0.711, both between 0 and 1. They are similar, so the OOB score gives a rough estimate of the validation score without holding data back.


### Exercise 10.4 (In class)

**Task.** Fit a random forest with 200 trees and a minimum of 3 houses per leaf, and read its out-of-bag R². Use `RandomForestRegressor(n_estimators=200, max_features=1/3, min_samples_leaf=3, oob_score=True, random_state=0, n_jobs=-1)` fitted on `Xtr, ytr`.

**Inputs.** `Xtr, ytr`.

**Define.** `oob_r2` (a number between 0 and 1: the forest's `oob_score_`).

**Form of the answer.** One number. The check allows a small tolerance because a forest is random.


In [ ]:
# Exercise 10.4: define oob_r2
oob_r2 = None


In [ ]:
assert abs(oob_r2 - 0.6525) < 0.02, "oob_r2 is not within 0.02 of the right value"
print("ok  oob_r2")


### Computing corner: forests run in parallel, boosting does not

Each tree of a forest is grown on its own bootstrap sample. No tree needs another tree. So the work can be split across processors (`n_jobs=-1` means "use all cores"). Below we time the same forest on one core and on all cores.

Boosting (next section) is different. Tree number k is fitted to the mistakes left by trees 1 to k − 1. It must wait for them. The trees themselves form a chain, so scikit-learn's boosting has no `n_jobs`. (Other libraries may parallelize the work inside one tree.)


In [ ]:
import os

def fit_forest(jobs):
    RandomForestRegressor(n_estimators=300, max_features=1 / 3, random_state=0, n_jobs=jobs).fit(Xtr, ytr)

time_1 = best_of(lambda: fit_forest(1), repeat=2)
time_all = best_of(lambda: fit_forest(-1), repeat=2)
n_cores = os.cpu_count()
print(f"this computer reports {n_cores} cores")
print(f"300 trees on 1 core:  {time_1:.2f} s")
print(f"300 trees on all cores: {time_all:.2f} s")
print(f"speed-up: {time_1 / time_all:.1f}x")


**Reading.** Read the number of cores printed above. Compare the three printed numbers above: the time on one core, the time on all cores, and their ratio. Timings change from run to run and from machine to machine. The speed-up is below the core count because starting workers and collecting results costs time too.


**Check yourself**

1. Why does averaging many trees reduce variance but not bias?
2. A forest has 300 trees. About how many of them have not seen a given row?

<details><summary>Answer</summary>

1. This is the main mechanism, not a law for every method. Each deep tree has low bias but a noisy fit. Averaging cancels noise that differs between trees. If all trees make the same systematic error, the average keeps it.
2. About 37% of 300, so about 110 trees.

</details>


## 5. Boosting, briefly

**Boosting** builds the model in a chain.

1. Start with a constant prediction, the mean price.
2. Fit a small tree to the **residuals** (price minus the current prediction).
3. Add a fraction of that tree to the prediction. The fraction is the **learning rate**.
4. Repeat. Each new small tree fixes some mistakes of the sum so far.

Each tree alone is weak (here depth 2), but a hundred of them together fit much better than one tree. The cell fits living area alone and shows the fit after 1, 10 and 100 rounds.


In [ ]:
gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=0).fit(area_tr, ytr)
staged = list(gb.staged_predict(grid))
fig, ax = plt.subplots()
ax.scatter(area_tr, ytr / 1000, s=3, color="lightgray")
for rounds, style, color in ((1, ":", "C1"), (10, "--", "C3"), (100, "-", "C0")):
    ax.plot(grid, staged[rounds - 1] / 1000, style, color=color, lw=1.8, label=f"after {rounds} round" + ("s" if rounds > 1 else ""))
ax.set_xlabel("living area (square feet)")
ax.set_ylabel("price (thousand US dollars)")
ax.set_ylim(0, 800)
ax.legend()
plt.tight_layout()
plt.show()
train_staged = [rmse(ytr, p) for p in gb.staged_predict(area_tr)]
print(f"training RMSE after 1 round {train_staged[0]:,.0f}, after 10 rounds {train_staged[9]:,.0f}, after 100 rounds {train_staged[99]:,.0f}")


**Reading.** After one round the curve is a single small step. After 10 rounds it follows the trend. After 100 rounds it follows the trend more closely. The training RMSE falls from 95,162 to 75,510 to 64,228 dollars. The learning rate of 0.1 means each tree only moves the prediction a tenth of the way, so many rounds are needed.


Now all features: 300 rounds of depth-3 trees against the forest of Section 4. Boosting is a chain, so there is no `n_jobs`.


In [ ]:
t0 = perf_counter()
gb_full = GradientBoostingRegressor(n_estimators=300, learning_rate=0.1, max_depth=3, random_state=0).fit(Xtr, ytr)
gb_time = perf_counter() - t0
gb_val = rmse(yva, gb_full.predict(Xva))
print(f"boosting (300 rounds, depth 3): validation RMSE {gb_val:,.0f}, fit time {gb_time:.2f} s")
print(f"random forest (300 trees):      validation RMSE {forest_val:,.0f}")
print(f"best single tree:               validation RMSE {best_tree_val:,.0f}")


**Reading.** Validation RMSE: boosting 50,751, forest 51,438, one tree 60,659. Both ensembles beat the single tree.


**Check yourself**

1. Bagging uses deep trees. Boosting uses shallow trees. Which method mainly attacks variance, and which mainly attacks bias?
2. Why can a forest be fitted on many cores but boosting cannot?

<details><summary>Answer</summary>

1. Mainly, bagging (and random forests) attack variance: deep trees have low bias and high variance, and averaging cuts the variance. Boosting mainly attacks bias: shallow trees have high bias, and each round reduces what is left. Boosting with too many rounds can also overfit.
2. Forest trees are independent. A boosting tree is fitted to the residuals of all earlier trees.

</details>

*Reading, not coding.* The book also shows BART (Bayesian additive regression trees), a boosting-like sum of small trees with a prior on each tree. It is used for causal questions, such as estimating how an effect varies across people. We do not fit it here.


## 6. Networks begin: the circle, a hidden layer, a forward pass

Trees cut the space into boxes. A neural network builds new features and then draws a line in the feature space. Recall the circle data: 200 points in the plane, class 0 near the center and class 1 in the outer ring around it. We use this convention everywhere: 0 is the center, 1 is the outer ring. A straight line fails, and in Part 2 you fixed it by hand with a new feature phi (the squared distance from the center). A network can build such a feature itself. Here the weights are set by hand so that you can follow every step. Nothing is trained in this lab; training comes later (Lab 12).


In [ ]:
circle = pd.read_csv(DATA / "circle.csv")
Xcirc = circle[["x1", "x2"]].to_numpy()
ycirc = circle.label.to_numpy()
line_acc = LogisticRegression().fit(Xcirc, ycirc).score(Xcirc, ycirc)
phi = (Xcirc**2).sum(axis=1)
phi_acc = LogisticRegression().fit(phi.reshape(-1, 1), ycirc).score(phi.reshape(-1, 1), ycirc)
plt.scatter(Xcirc[ycirc == 0, 0], Xcirc[ycirc == 0, 1], s=12, color="C0", label="class 0 (center)")
plt.scatter(Xcirc[ycirc == 1, 0], Xcirc[ycirc == 1, 1], s=12, color="C3", marker="x", label="class 1 (outer ring)")
plt.xlabel("x1")
plt.ylabel("x2")
plt.gca().set_aspect("equal")
plt.legend(loc="upper right", fontsize=7)
plt.tight_layout()
plt.show()
print(f"{len(circle)} points; a straight line gets accuracy {line_acc:.3f}; the hand-made feature phi gets {phi_acc:.3f}")


**Reading.** The straight line scores 0.525, no better than a coin flip, because the classes are nested, not side by side. On these 200 points the feature phi separates them completely (accuracy 1.000; this is accuracy on the displayed points, not evidence about new data). Can a network find something similar?


### The network, with given weights

A **hidden layer** computes several new features from the inputs. Each hidden unit takes a weighted sum of the inputs, adds a bias, and applies the **ReLU** function, relu(z) = max(0, z). The output unit then takes a weighted sum of the hidden values.

Online, the [Neural Net Playground](https://vsokolov.org/labs/neural-net-lab.html) lets you set the weights by hand in the playground, then watch it train.

Our network has 2 inputs, 4 hidden units, and 1 output (a 2-4-1 network). Its weights are given, not learned.

- Hidden unit 1 fires when x1 is above 4. Unit 2 fires when x1 is below −4. Units 3 and 4 do the same for x2.
- So the hidden values together measure how far a point sticks out of the square from −4 to 4.
- The output adds the four hidden values and subtracts 0.5. A positive output means class 1. So class 1 starts where the sum of the hidden values exceeds 0.5: on the right, where x1 − 4 > 0.5, that is x1 > 4.5.

The weights are stored in four arrays:

- `W1` has shape 2 by 4: rows are inputs, columns are hidden units.
- `b1` holds 4 biases, one per hidden unit.
- `W2` holds 4 weights, one per hidden unit.
- `b2` is one number, the output bias.

In symbols, with row-vector input x: h = relu(x W1 + b1) and output = h W2 + b2.


In [ ]:
W1 = np.array([[1.0, -1.0, 0.0, 0.0],
               [0.0, 0.0, 1.0, -1.0]])   # 2 inputs -> 4 hidden units
b1 = np.array([-4.0, -4.0, -4.0, -4.0])   # one bias per hidden unit
W2 = np.array([1.0, 1.0, 1.0, 1.0])       # 4 hidden units -> 1 output
b2 = -0.5                                 # one bias for the output

# Predict first: which hidden unit fires for (5, -1), and which class results?
# Forward pass by hand for the point (5, -1)
x_pt = np.array([5.0, -1.0])
z1 = x_pt[0] * W1[0, 0] + x_pt[1] * W1[1, 0] + b1[0]
z2 = x_pt[0] * W1[0, 1] + x_pt[1] * W1[1, 1] + b1[1]
z3 = x_pt[0] * W1[0, 2] + x_pt[1] * W1[1, 2] + b1[2]
z4 = x_pt[0] * W1[0, 3] + x_pt[1] * W1[1, 3] + b1[3]
print("hidden sums z:", z1, z2, z3, z4)
h_pt = np.maximum([z1, z2, z3, z4], 0.0)
print("after ReLU h: ", h_pt)
out_pt = h_pt[0] * W2[0] + h_pt[1] * W2[1] + h_pt[2] * W2[2] + h_pt[3] * W2[3] + b2
print("output:", out_pt, "->", "class 1" if out_pt > 0 else "class 0")


**Reading.** For the point (5, −1): the four sums are 1, -9, -5, -3. ReLU keeps only the first (it equals 1) and sets the others to 0. The output is 0.5, which is positive, so the network says class 1. Only unit 1 fired, because the point sticks out of the square on the right.


### The same pass for all 200 points: one matrix product

Instead of looping, stack the points as rows of a matrix X. Then the whole layer is one product. **Predict** the shape of each array before you run the cell (the first one is given: X is n × 2, here 200 × 2).


In [ ]:
n_points = len(Xcirc)
A1 = Xcirc @ W1 + b1            # (n,2) @ (2,4) -> (n,4), then b1 (4,) is added to every row
H = np.maximum(A1, 0.0)         # (n,4)
out = H @ W2 + b2               # (n,4) @ (4,) -> (n,)
pred_net = (out > 0).astype(int)
net_acc = float((pred_net == ycirc).mean())
print("X  ", Xcirc.shape, "  n x 2   (batch axis first)")
print("W1 ", W1.shape, "    2 x 4")
print("b1 ", b1.shape, "      4")
print("A1 ", A1.shape, "  n x 4")
print("H  ", H.shape, "  n x 4")
print("W2 ", W2.shape, "      4")
print("out", out.shape, "    n")
print(f"accuracy of the network on the 200 displayed points: {net_acc:.3f}")
print("parameters:", W1.size, "+", b1.size, "+", W2.size, "+ 1 =", W1.size + b1.size + W2.size + 1)
n_params = W1.size + b1.size + W2.size + 1


**Reading.** Every array has a shape and the shapes must match along the shared axis: (n × 2) times (2 × 4) gives (n × 4). The batch axis (n) comes first and is untouched. On the 200 displayed points the network has accuracy 1.000, where the straight line got 0.525. This shows what the network can represent. It is accuracy on the displayed points, not independent evidence that the network would generalize. The parameters are 8 weights in W1, 4 biases in b1, 4 weights in W2 and 1 bias in b2, which makes 17.


In [ ]:
# Your turn: move one weight and see which part of the boundary moves.
# Target: with the first bias at -2, the right-hand edge of class 1 moves from x1 = 4.5 to x1 = 2.5.
# Then try -6. To change a weight in W1, change a COPY (W1_try = W1.copy()) and never edit W1, b1, W2, b2 themselves: Exercise 10.5 uses them.
b1_new = b1.copy()
b1_new[0] = -4.0  # change this (try -2.0)

def net_class(points, W1, b1, W2, b2):
    return ((np.maximum(points @ W1 + b1, 0.0) @ W2 + b2) > 0).astype(int)

gx, gy = np.meshgrid(np.linspace(-10, 10, 200), np.linspace(-10, 10, 200))
gpts = np.c_[gx.ravel(), gy.ravel()]
fig, ax = plt.subplots()
ax.contour(gx, gy, net_class(gpts, W1, b1, W2, b2).reshape(gx.shape), levels=[0.5], colors="C0", linestyles="solid")
ax.contour(gx, gy, net_class(gpts, W1, b1_new, W2, b2).reshape(gx.shape), levels=[0.5], colors="C3", linestyles="dashed")
ax.scatter(Xcirc[:, 0], Xcirc[:, 1], c=np.where(ycirc == 1, "C3", "C0"), s=6)
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_aspect("equal")
ax.set_title("boundary: solid = given, dashed = changed", fontsize=8)
plt.tight_layout()
plt.show()
print("accuracy with the changed bias:", (net_class(Xcirc, W1, b1_new, W2, b2) == ycirc).mean())
acc_minus2 = float((net_class(Xcirc, W1, np.array([-2.0, -4.0, -4.0, -4.0]), W2, b2) == ycirc).mean())
print("accuracy with the first bias at -2:", acc_minus2)


**Predict:** if we remove ReLU from every hidden unit (so the hidden values are just x W1 + b1), can the network still separate the ring from the center? Write your guess and a reason, then run the cell.


In [ ]:
A1_lin = Xcirc @ W1 + b1               # hidden values with no ReLU
out_lin = A1_lin @ W2 + b2
W_combined = W1 @ W2                   # the two layers collapse into one: shape (2,)
b_combined = float(b1 @ W2 + b2)
wc0, wc1 = float(W_combined[0]), float(W_combined[1])
no_relu_acc = float(((out_lin > 0).astype(int) == ycirc).mean())
no_relu_value = float(out_lin.mean())
print("W1 @ W2 =", W_combined, " b1 @ W2 + b2 =", b_combined)
print("distinct output values without ReLU:", np.unique(np.round(out_lin, 6)))
print(f"accuracy without ReLU on the 200 displayed points: {no_relu_acc:.3f}")


**Reading.** Without ReLU the two layers collapse into one linear map: out = x (W1 W2) + (b1 W2 + b2). Here W1 W2 = (0, 0). With these weights, removing ReLU makes every prediction -16.5, so accuracy is 0.500, no better than a coin flip. More generally, a network made entirely of affine layers is equivalent to one affine layer, however many layers it has: it can still change with the input, but only along a straight line, so it cannot separate the center from the surrounding ring. The ReLU is what lets the hidden layer bend the boundary.


In [ ]:
# Reset: Exercise 10.5 uses these fixed weights (the Your turn cell only changed copies).
W1 = np.array([[1.0, -1.0, 0.0, 0.0],
               [0.0, 0.0, 1.0, -1.0]])
b1 = np.array([-4.0, -4.0, -4.0, -4.0])
W2 = np.array([1.0, 1.0, 1.0, 1.0])
b2 = -0.5
x_new = np.array([-6.0, 5.5])
shift = np.linspace(0.0, 1.0, 200)


### Exercise 10.5 (In class)

**Task.** Run the forward pass for a new point and for all the circle points, and repair one shape error. The cell above this one resets the fixed weights, so use `W1, b1, W2, b2` as they are.

**Inputs.** `x_new = np.array([-6.0, 5.5])`, `shift = np.linspace(0.0, 1.0, 200)` (one number for each example), `Xcirc`, `ycirc`, and `W1, b1, W2, b2`. There is no `relu` function: use `np.maximum(..., 0.0)`.

**Define.**
- `y_new`: a plain number, the network output for `x_new`.
- `H_all`: the hidden matrix for all points, shape (200, 4).
- `acc_circle`: predict class 1 when the output is above 0, otherwise class 0 (class 0 is the center, class 1 the outer ring); `acc_circle` is the share of the 200 points whose predicted class equals `ycirc`.
- `A1_rows`: shape (200, 4). Start from `Xcirc @ W1` and add to all four hidden sums of example i the number `shift[i]`. The line `Xcirc @ W1 + shift` fails with a shape error; repair it. (Do not add b1 here.)

**Form of the answer.** One number, one array of shape (200, 4), one number, one array of shape (200, 4). Then answer the written question in the cell below the check.


In [ ]:
# Exercise 10.5: define y_new, H_all, acc_circle, A1_rows
y_new = None
H_all = None
acc_circle = None
A1_rows = None


In [ ]:
check("y_new", y_new, "413f1f4a150e4f67")
check_array("H_all", H_all, "fb20f8d782070088", (200, 4))
check_array("A1_rows", A1_rows, "e78cf80ff9ca0d4e", (200, 4))
check("acc_circle", acc_circle, "fa5f65487538c694")


**Written (2 or 3 sentences).** If we remove ReLU from all four hidden units, can the network still separate the center (class 0) from the outer ring (class 1)? Use the cell above to check.

YOUR ANSWER HERE


### Computing corner: tensor mechanics, part 1

Every array has a **shape** (the length along each **axis**). Every operation has a rule for shapes. Most bugs in deep learning code are shape bugs, so we practise the rules on the arrays above.

**Axes.** `H.sum(axis=0)` adds down the rows and leaves one number per column (one per hidden unit). `H.sum(axis=1)` adds across the columns and leaves one number per row (one per example). **Predict** the two shapes, then run.


In [ ]:
per_unit = H.sum(axis=0)
per_example = H.sum(axis=1)
print("H.sum(axis=0):", per_unit.shape, per_unit)
print("H.sum(axis=1):", per_example.shape, "first three:", per_example[:3])


**Reading.** `axis=0` removes the first axis, so the result has shape (4,): one number per hidden unit. `axis=1` removes the second axis, so the result has shape (200,): one number per example. The rule: the axis you name disappears.


**Broadcasting.** Adding b1 (shape (4,)) to the (n × 4) product works with no loop. NumPy lines the shapes up from the right and stretches any axis of length 1 (or a missing axis) to match. A bias of shape (n,) does not line up: it fails when n is not 4. Worse, when n happens to equal 4, it runs and silently adds the wrong thing. **Predict** the result shape of each addition, then run. (We add the bias to `A_pre`, the product before the bias, so that we never add b1 twice.)


In [ ]:
A_pre = Xcirc @ W1   # the product before any bias, shape (n, 4); A1 already contains b1, so we never add b1 to A1 again
print("A_pre + b1 with b1 of shape (4,):", (A_pre + b1).shape)
try:
    A_pre + np.ones(len(A_pre))
except ValueError as err:
    print("a bias of shape (n,):", "ValueError:", err)

Z4 = A_pre[:4]                           # pretend the batch has exactly 4 examples
per_example_bias = np.array([0.0, 10.0, 20.0, 30.0])   # one bias per EXAMPLE (the wrong idea)
wrong = Z4 + per_example_bias         # runs, but adds 0, 10, 20, 30 to the four COLUMNS
right = Z4 + per_example_bias[:, None]   # shape (4,1) adds to the four rows
print("batch of 4, no error but different answers:", not np.allclose(wrong, right))
print("wrong adds the same to every row:", np.allclose(wrong - Z4, per_example_bias))


**Reading.** The (4,) bias broadcasts to shape (200, 4) and works. The (n,) bias fails with a ValueError. With a batch of exactly 4 the wrong bias runs without any warning and gives a different answer from the intended one. To add one number per row, give it shape (n, 1) with `[:, None]`. Always print shapes when a result looks odd.


**Reshape.** A dense layer wants each example as one row of numbers. A batch of 28 × 28 images has shape (n, 28, 28). We flatten each image to 784 numbers and back. (The images here are simulated random numbers; only the shapes matter.)


In [ ]:
images = np.random.default_rng(1).random((64, 28, 28))
flat = images.reshape(len(images), -1)     # -1 means "work out this length yourself"
back = flat.reshape(-1, 28, 28)
print("images:", images.shape, "-> flat:", flat.shape, "-> back:", back.shape)
print("round trip is exact:", np.array_equal(images, back))


**Reading.** (64, 28, 28) becomes (64, 784) because 28 × 28 = 784. The batch axis is kept, and reshaping back restores the images exactly. Reshape never changes the numbers or their order, only how we cut them into axes.


### Computing corner: what a matrix product costs

For a product of an (n × d) matrix and a (d × h) matrix, each of the n × h outputs needs d multiplications and d additions. The cost is about **2 n d h floating-point operations** (FLOPs). A forward pass for a batch is one such product per layer.

We time a triple Python loop against one `X @ W` for n = 1,000, d = 20, h = 32. The two give the same answer.


In [ ]:
rng_mm = np.random.default_rng(2)
n_mm, d_mm, h_mm = 1000, 20, 32
Xm = rng_mm.standard_normal((n_mm, d_mm))
Wm = rng_mm.standard_normal((d_mm, h_mm))

def triple_loop(X, W):
    out = np.zeros((X.shape[0], W.shape[1]))
    for i in range(X.shape[0]):
        for j in range(W.shape[1]):
            for k in range(X.shape[1]):
                out[i, j] += X[i, k] * W[k, j]
    return out

loop_result = triple_loop(Xm, Wm)
time_loop = best_of(lambda: triple_loop(Xm, Wm), repeat=1)
time_vec = best_of(lambda: Xm @ Wm, repeat=20)
flops_mm = 2 * n_mm * d_mm * h_mm
print("same answer:", np.allclose(loop_result, Xm @ Wm))
print(f"FLOPs: 2 * {n_mm} * {d_mm} * {h_mm} = {flops_mm:,}")
print(f"triple loop: {time_loop:.3f} s   X @ W: {1000 * time_vec:.3f} ms   ratio {time_loop / time_vec:,.0f}x")
print(f"X @ W runs at about {flops_mm / time_vec / 1e9:.1f} GFLOP/s")


**Reading.** The product needs 1,280,000 FLOPs. The printed times show that the loop is thousands of times slower than `X @ W` (the exact ratio changes from run to run). The gap comes from compiled, cache-friendly code that uses vector instructions and several cores. This is the case for **vectorization**: write the whole batch as one array operation.


### Computing corner: CPU against GPU

A GPU has thousands of small cores. It is fast at big matrix products and slow at small jobs. We time an n × n float32 matrix product with NumPy on the CPU and with PyTorch on the best device available: `cuda` (Nvidia GPU, as on Colab), else `mps` (Apple silicon GPU), else `cpu`. If this computer has no GPU, the second column is PyTorch on the CPU, and the numbers will differ from the instructor's table below.

GPU calls return before the work is done. So we call `synchronize()` before stopping the clock; without it we would time only the launch.

**Predict:** at what size n does the GPU start to beat the CPU?


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

def sync():
    if device == "cuda":
        torch.cuda.synchronize()
    elif device == "mps":
        torch.mps.synchronize()

def time_numpy(n):
    a = np.random.default_rng(0).standard_normal((n, n)).astype(np.float32)
    b = np.random.default_rng(1).standard_normal((n, n)).astype(np.float32)
    a @ b
    return best_of(lambda: a @ b, repeat=3)

def time_torch(n):
    a = torch.randn(n, n, device=device, dtype=torch.float32)
    b = torch.randn(n, n, device=device, dtype=torch.float32)
    a @ b
    sync()
    def run():
        a @ b
        sync()
    return best_of(run, repeat=3)

sizes = [256, 1024, 2048, 4096]
cpu_times = [time_numpy(n) for n in sizes]
dev_times = [time_torch(n) for n in sizes]
print("device used by PyTorch:", device)
print(f"{'n':>6} {'CPU (NumPy)':>14} {'PyTorch on ' + device:>18} {'CPU / device':>14}")
for n, tc, tg in zip(sizes, cpu_times, dev_times):
    print(f"{n:>6} {1000 * tc:>11.2f} ms {1000 * tg:>15.2f} ms {tc / tg:>13.2f}x")

plt.loglog(sizes, cpu_times, "C0-o", label="CPU (NumPy)")
plt.loglog(sizes, dev_times, "C3--s", label=f"PyTorch on {device}")
plt.xlabel("matrix size n")
plt.ylabel("time (seconds)")
plt.legend()
plt.tight_layout()
plt.show()


**Reading.** Read the device printed above. Each doubling of n multiplies the work by 8 (2 n³ FLOPs), so both lines rise steeply on the log scales. Compare the ratio column across sizes to see where the device starts to win or lose on your computer. The instructor's recorded numbers follow; compare them with your own printed table.


In [ ]:
# Launch overhead and copy cost
x_small_np = np.ones(10, dtype=np.float32)
x_small = torch.ones(10, device=device)

def tiny_cpu():
    x = x_small_np
    for _ in range(1000):
        x = x + 1

def tiny_dev():
    x = x_small
    for _ in range(1000):
        x = x + 1
    sync()

tiny_cpu_time = best_of(tiny_cpu)
tiny_dev_time = best_of(tiny_dev)
print(f"1,000 tiny operations: NumPy {1000 * tiny_cpu_time:.2f} ms, PyTorch on {device} {1000 * tiny_dev_time:.2f} ms")

big = torch.empty(64 * 1024 * 1024, dtype=torch.float32)     # 256 MB on the CPU
t0 = perf_counter()
big_on_device = big.to(device)
sync()
copy_time = perf_counter() - t0
print(f"copying {big.numel() * 4 / 2**20:.0f} MB to {device}: {copy_time:.3f} s" + ("  (no copy needed: already on the CPU)" if device == "cpu" else ""))
del big_on_device


**The instructor's numbers.** Measured on the instructor's Apple M2 laptop on 22 October (CPU = NumPy through Apple's Accelerate library, GPU = PyTorch on `mps`). Your numbers differ from run to run.

| n | CPU (NumPy) | GPU (MPS) |
|---|---|---|
| 256 | 0.03 ms | 0.3 ms |
| 1024 | 2.1 ms | 2.6 ms |
| 2048 | 27 ms | 7.6 ms |
| 4096 | 192 ms | 48 ms |
| 8192 | 1.09 s | 0.68 s |

| Other measurement | CPU | GPU |
|---|---|---|
| 1,000 tiny operations | 2.2 ms | 6.4 ms |
| copying 256 MB to the GPU (instructor's laptop) | not needed | 0.39 s |

**Four lessons.**

1. On the instructor's laptop the GPU starts to win between n = 1,024 and n = 2,048. At n = 2048 it is about 3.6 times faster (27 ms against 7.6 ms), and at n = 4096 about 4 times (192 ms against 48 ms).
2. Small jobs lose on the GPU because every call has a fixed launch cost. At n = 256 the GPU took 0.3 ms against 0.03 ms. For 1,000 tiny operations the GPU took 6.4 ms and the CPU 2.2 ms.
3. Moving data costs time. On the instructor's laptop, copying 256 MB to the GPU took 0.39 s, longer than one 4096 × 4096 product (48 ms). Keep data on the device and copy as rarely as possible.
4. The M2 CPU is already fast: 2 × 4096³ FLOPs in 192 ms is about 0.7 TFLOP/s, and at n = 8192 the CPU reaches about 1 TFLOP/s. The laptop GPU wins by at most about 4 times (at n = 8192 only about 1.6 times: 1.09 s against 0.68 s), not 100. Big gaps appear with data-center GPUs (see the Hopper A100 table below).


**Recorded on a data-center GPU.** One NVIDIA A100 (80 GB) on GMU's Hopper cluster, run on 22 October 2026, against 16 CPU cores of the same server. Seconds for one n × n matrix product. TF32 and bfloat16 are lower-precision number formats that the GPU computes faster. TF32 is a format Nvidia GPUs use inside matrix products: it keeps the range of float32 but fewer digits of precision, which makes the product faster. Lab 12 explains bfloat16.

| n | CPU, 16 cores, float32 | A100, float32 | A100, TF32 | A100, bfloat16 |
|---|---|---|---|---|
| 256 | 0.52 ms | 0.014 ms | 0.016 ms | 0.011 ms |
| 1,024 | 2.06 ms | 0.13 ms | 0.033 ms | 0.020 ms |
| 2,048 | 13 ms | 0.98 ms | 0.17 ms | 0.11 ms |
| 4,096 | 94 ms | 7.26 ms | 1.05 ms | 0.52 ms |
| 8,192 | 716 ms | 58 ms | 7.87 ms | 3.81 ms |

- Best speed measured: CPU 1.5 TFLOP/s; A100 19 TFLOP/s in float32, 140 in TF32, 291 in bfloat16. (1 TFLOP/s is a trillion floating-point operations per second.)
- At n = 8,192 the A100 is 12 times faster than the server CPU in float32 and 188 times faster in bfloat16. Unlike the laptop above, this GPU already wins at n = 256.
- The overheads are the same as on the laptop: 1,000 tiny operations took 6.5 ms on the GPU against 3.2 ms on the CPU, and copying 256 MB to the GPU took 19 ms (13 GB/s), about 5 times as long as one 8,192 bfloat16 product. Keep the data on the GPU and give it big operations.


## Exit problem (paper, no code)

**(a)** Six points have x = 1, 2, 3, 4, 5, 6 and y = 2, 3, 2, 8, 9, 10. Compute the SSE of the split "x < 3.5" and of the split "x < 2.5". Which split is better?

**(b)** A network has 2 inputs, 2 ReLU hidden units, and 1 output, with

W1 = [[1, −2], [2, 1]] (rows are inputs, columns are hidden units), b1 = [−1, −1], W2 = [0.5, −1], b2 = 0.5.

Compute h = relu(x W1 + b1) and the output for x = (1, 2). Write the shape of x, W1, b1, h, W2 and the output.


## At home

Finish any in-class exercise you did not complete. Then do Exercise 10.6.


### Exercise 10.6 (At home)

**Task.** Count the floating-point operations of one forward pass through the layers 784 to 256 to 10 for a batch of 64 examples. Count only the two matrix products, each as 2 × n × d × h. (Ignore the bias additions and the ReLU.)

**Inputs.** The batch size 64 and the layer sizes 784, 256, 10.

**Define.** `flops` (an integer: the total for both products).

**Form of the answer.** One integer.


In [ ]:
# Exercise 10.6: define flops
flops = None


In [ ]:
check("flops", flops, "fca1d44e42d784f9")


## Before you submit

- Restart the kernel and run every cell. Each check cell prints `ok`.
- Every exercise cell is filled in (no `None` left).
- Cheat sheet check. Without looking, can you write: the SSE of a group and the score of a split; the Gini index of a node; what cost-complexity pruning minimizes; how a bootstrap sample and the out-of-bag rows work; which of bagging and boosting attacks variance and which bias; the shapes in h = relu(x W1 + b1), output = h W2 + b2; the FLOP count 2ndh; why the crossover size, where a GPU starts to win, depends on the hardware, the software, the size of the operation, and the cost of moving data? Whatever you cannot write goes on the sheet.
